In [26]:
# ==============================================================================
# 🎯 MOTOR PREDITIVO: ESCANEAMENTO DAS 17:00 (INTEGRAÇÃO RELACIONAL NEONDB)
# ==============================================================================
import pandas as pd
from sqlalchemy import select
from datetime import date, timedelta
from database import AsyncSessionLocal
from models import ResultadoLoteria, BichoGrupoDezena  # Chamei nossa nova tabela aqui!

# 1. Puxa os dados atualizados com o relacionamento direto do banco
async with AsyncSessionLocal() as session:
    # 🐾 Adeus dicionário fixo! Buscamos a tabela de bichos viva do NeonDB
    bichos_query = await session.execute(select(BichoGrupoDezena))
    tabela_bichos_db = {b.grupo: b.bicho for b in bichos_query.scalars().all()}
    
    # 📊 Busca os resultados das loterias
    res = await session.execute(select(ResultadoLoteria))
    df = pd.DataFrame([item.__dict__ for item in res.scalars().all()])

# Tratamento e Higienização do DataFrame usando os dados dinâmicos do banco
df['data_hora'] = pd.to_datetime(df['data_hora'])
df['data_sorteio'] = df['data_hora'].dt.date

# 🔗 SACADA DE ENGENHARIA DE DADOS:
# Como a coluna 'grupo' já vem pronta do banco de dados, fazemos apenas o mapeamento direto!
# (Usamos um fallback de segurança caso existam dados antigos com grupo nulo)
if 'grupo' in df.columns:
    # Preenche nulos antigos se houver, convertendo a dezena na marra
    df['grupo'] = df['grupo'].fillna(
        df['resultado'].astype(str).str.zfill(2).str[-2:].apply(
            lambda d: 25 if int(d) == 0 else ((int(d) - 1) // 4) + 1 if d.isdigit() else None
        )
    )
else:
    # Se por acaso a coluna sumir, calcula dinamicamente
    df['grupo'] = df['resultado'].astype(str).str.zfill(2).str[-2:].apply(
        lambda d: 25 if int(d) == 0 else ((int(d) - 1) // 4) + 1 if d.isdigit() else None
    )

# Faz a tradução mágica usando o dicionário gerado pelo NeonDB
df['bicho'] = df['grupo'].map(tabela_bichos_db)

# Definição das variáveis de tempo
hoje_analise = date.today()
data_60_dias = hoje_analise - timedelta(days=60)

print(f"🔮 PROJEÇÃO ESTATÍSTICA: NACIONAL 17:00 (Matriz Atualizada via DB) 🔮")
print("=" * 80)

# FILTRO A: Memória Específica do Horário (Últimos 60 dias às 17:00)
df_17h_hist = df[(df['no_loteria'] == 'Nacional') & 
                 (df['horario'] == '17:00') & 
                 (df['data_sorteio'] >= data_60_dias)]
contagem_17h_hist = df_17h_hist['bicho'].value_counts()

print("⏳ 1. MEMÓRIA DO HORÁRIO (Donos históricos das 17:00 nos últimos 60 dias):")
for bicho, qtd in contagem_17h_hist.head(3).items():
    print(f"   • {bicho:<12} ➔ {qtd} aparições específicas neste horário")

# FILTRO B: Comportamento das últimas 24h pós-15:00
limite_24h = df['data_hora'].max() - pd.Timedelta(hours=24)
df_24h = df[(df['no_loteria'] == 'Nacional') & (df['data_hora'] >= limite_24h)]
contagem_24h = df_24h['bicho'].value_counts()

print("-" * 80)
print("⚡ 2. MOMENTUM RECENTE (O que mais saiu nas últimas 24h incluindo as 15:00):")
for bicho, qtd in contagem_24h.head(3).items():
    print(f"   • {bicho:<12} ➔ {qtd} aparições recentes")

print("=" * 80)
# FILTRO C: Cruzamento Inteligente (Chaves de Convergência)
bichos_chave = set(contagem_17h_hist.head(5).index).intersection(set(contagem_24h.head(5).index))
print("🎯 3. ANIMAIS DE CONVERGÊNCIA (União de Histórico das 17h + Força de Hoje):")
if bichos_chave:
    print(f"   🔥 ALVOS CRÍTICOS DETECTADOS: {', '.join(bichos_chave)}")
else:
    print("   ℹ️ Sem convergência direta. O algoritmo está alternando entre ciclos puros.")
print("=" * 80)

🔮 PROJEÇÃO ESTATÍSTICA: NACIONAL 17:00 (Matriz Atualizada via DB) 🔮
⏳ 1. MEMÓRIA DO HORÁRIO (Donos históricos das 17:00 nos últimos 60 dias):
   • PERU         ➔ 25 aparições específicas neste horário
   • CAMELO       ➔ 24 aparições específicas neste horário
   • BURRO        ➔ 24 aparições específicas neste horário
--------------------------------------------------------------------------------
⚡ 2. MOMENTUM RECENTE (O que mais saiu nas últimas 24h incluindo as 15:00):
   • COELHO       ➔ 7 aparições recentes
   • LEÃO         ➔ 5 aparições recentes
   • CABRA        ➔ 5 aparições recentes
🎯 3. ANIMAIS DE CONVERGÊNCIA (União de Histórico das 17h + Força de Hoje):
   🔥 ALVOS CRÍTICOS DETECTADOS: CABRA


In [28]:
# ==============================================================================
# 🐾 ETAPA 1: TERMÔMETRO HORÁRIO MULTIJANELA (24H vs 48H vs 72H)
# ==============================================================================
import pandas as pd
from sqlalchemy import select
from datetime import date, timedelta
from database import AsyncSessionLocal
from models import ResultadoLoteria, BichoGrupoDezena

# ⚙️ PARAMETRIZAÇÃO DO ALVO (Altere o horário conforme o dia avança)
HORARIO_ALVO = '17:00'
LOTERIA_ALVO = 'Nacional'

# 1. Carga Relacional Limpa do NeonDB
async with AsyncSessionLocal() as session:
    bichos_query = await session.execute(select(BichoGrupoDezena))
    tabela_bichos_db = {b.grupo: b.bicho for b in bichos_query.scalars().all()}
    
    res = await session.execute(select(ResultadoLoteria))
    df_raw = pd.DataFrame([item.__dict__ for item in res.scalars().all()])

# 2. Preparação da Matriz
df_raw['data_hora'] = pd.to_datetime(df_raw['data_hora'])
df_raw['data_sorteio'] = df_raw['data_hora'].dt.date
df_raw['dezena'] = df_raw['resultado'].astype(str).str.zfill(2).str[-2:]
df_raw['grupo'] = df_raw['grupo'].fillna(
    df_raw['resultado'].astype(str).str.zfill(2).str[-2:].apply(
        lambda d: 25 if int(d) == 0 else ((int(d) - 1) // 4) + 1 if d.isdigit() else None
    )
)
df_raw['bicho'] = df_raw['grupo'].map(tabela_bichos_db)

# Filtra estritamente a loteria e o horário alvo para evitar ruídos de outros turnos
df_alvo = df_raw[(df_raw['no_loteria'] == LOTERIA_ALVO) & (df_raw['horario'] == HORARIO_ALVO)].copy()

# 3. Definição das Janelas Temporais Móveis
ultimo_sorteio = df_raw['data_hora'].max()
marcador_24h = ultimo_sorteio - pd.Timedelta(hours=24)
marcador_48h = ultimo_sorteio - pd.Timedelta(hours=48)
marcador_72h = ultimo_sorteio - pd.Timedelta(hours=72)

janelas = {
    "⚡ 24 HORAS MÓVEIS": df_alvo[df_alvo['data_hora'] >= marcador_24h],
    "🔄 48 HORAS MÓVEIS": df_alvo[df_alvo['data_hora'] >= marcador_48h],
    "⏳ 72 HORAS MÓVEIS": df_alvo[df_alvo['data_hora'] >= marcador_72h]
}

print(f"🌡️ RASTREADOR TÉRMICO RELACIONAL - HORÁRIO: {HORARIO_ALVO} ({LOTERIA_ALVO}) 🌡️")
print("=" * 80)

for nome_janela, df_janela in janelas.items():
    print(f"\n{nome_janela}")
    print("-" * 80)
    
    if df_janela.empty:
        print("   -> Sem registros suficientes nesta janela para o horário alvo.")
        continue
        
    contagem = df_janela['bicho'].value_counts()
    
    # Cortes matemáticos baseados na distribuição real da janela
    q_alta = contagem.quantile(0.65)
    q_baixa = contagem.quantile(0.35)
    
    quentes, mornos, frios = [], [], []
    
    for grupo_id, bicho_nome in tabela_bichos_db.items():
        qtd = contagem.get(bicho_nome, 0)
        item_str = f"{bicho_nome} (G-{str(grupo_id).zfill(2)}) [{qtd}x]"
        
        if qtd >= q_alta and qtd > 0:
            quentes.append(item_str)
        elif qtd > q_baixa:
            mornos.append(item_str)
        else:
            frios.append(item_str)
            
    print(f"   🔥 QUENTES (Aceleração): {', '.join(quentes[:5]) if quentes else 'Nenhum'}")
    print(f"   🫖 MORNOS     (Estáveis): {', '.join(mornos[:5]) if mornos else 'Nenhum'}")
    print(f"   ❄️ FRIOS     (Retidos) : {', '.join(frios[:5]) if frios else 'Nenhum'}")
print("=" * 80)

🌡️ RASTREADOR TÉRMICO RELACIONAL - HORÁRIO: 17:00 (Nacional) 🌡️

⚡ 24 HORAS MÓVEIS
--------------------------------------------------------------------------------
   🔥 QUENTES (Aceleração): ÁGUIA (G-02) [1x], BORBOLETA (G-04) [1x], CABRA (G-06) [1x], CAMELO (G-08) [2x], COELHO (G-10) [1x]
   🫖 MORNOS     (Estáveis): Nenhum
   ❄️ FRIOS     (Retidos) : AVESTRUZ (G-01) [0x], BURRO (G-03) [0x], CACHORRO (G-05) [0x], CARNEIRO (G-07) [0x], COBRA (G-09) [0x]

🔄 48 HORAS MÓVEIS
--------------------------------------------------------------------------------
   🔥 QUENTES (Aceleração): ÁGUIA (G-02) [1x], BORBOLETA (G-04) [2x], CACHORRO (G-05) [1x], CABRA (G-06) [1x], CARNEIRO (G-07) [1x]
   🫖 MORNOS     (Estáveis): Nenhum
   ❄️ FRIOS     (Retidos) : AVESTRUZ (G-01) [0x], BURRO (G-03) [0x], COBRA (G-09) [0x], CAVALO (G-11) [0x], GALO (G-13) [0x]

⏳ 72 HORAS MÓVEIS
--------------------------------------------------------------------------------
   🔥 QUENTES (Aceleração): AVESTRUZ (G-01) [1x], ÁGU

In [30]:
# ==============================================================================
# 👥 ETAPA 2: RASTREADOR DE CASAIS (COOCORRÊNCIA EM SORTEIOS DAS 17:00)
# ==============================================================================
import pandas as pd
from sqlalchemy import select
from datetime import date, timedelta
from itertools import combinations  # Maquinário matemático para gerar os pares
from database import AsyncSessionLocal
from models import ResultadoLoteria, BichoGrupoDezena

HORARIO_ALVO = '17:00'
LOTERIA_ALVO = 'Nacional'

# 1. Puxa tabelas vivas do banco de dados
async with AsyncSessionLocal() as session:
    bichos_query = await session.execute(select(BichoGrupoDezena))
    tabela_bichos_db = {b.grupo: b.bicho for b in bichos_query.scalars().all()}
    
    res = await session.execute(select(ResultadoLoteria))
    df_raw = pd.DataFrame([item.__dict__ for item in res.scalars().all()])

# 2. Modelagem e Higienização dos dados na RAM
df_raw['data_hora'] = pd.to_datetime(df_raw['data_hora'])
df_raw['grupo'] = df_raw['grupo'].fillna(
    df_raw['resultado'].astype(str).str.zfill(2).str[-2:].apply(
        lambda d: 25 if int(d) == 0 else ((int(d) - 1) // 4) + 1 if d.isdigit() else None
    )
)
df_raw['bicho'] = df_raw['grupo'].map(tabela_bichos_db)

# Filtra estritamente a loteria e o horário alvo
df_alvo = df_raw[(df_raw['no_loteria'] == LOTERIA_ALVO) & (df_raw['horario'] == HORARIO_ALVO)].copy()

# 3. Marcadores Temporais Móveis
ultimo_sorteio = df_raw['data_hora'].max()
janelas = {
    "⚡ JANELA 24H MÓVEIS": df_alvo[df_alvo['data_hora'] >= (ultimo_sorteio - pd.Timedelta(hours=24))],
    "🔄 JANELA 48H MÓVEIS": df_alvo[df_alvo['data_hora'] >= (ultimo_sorteio - pd.Timedelta(hours=48))],
    "⏳ JANELA 72H MÓVEIS": df_alvo[df_alvo['data_hora'] >= (ultimo_sorteio - pd.Timedelta(hours=72))]
}

def calcular_frequencia_casais(df_janela):
    """Agrupa por sorteio e calcula quais pares saíram juntos no mesmo resultado"""
    # Agrupa por cada sorteio individual e gera um set de bichos únicos que saíram nele
    sorteios = df_janela.groupby('data_hora')['bicho'].apply(lambda x: list(set(x.dropna())))
    
    lista_pares = []
    for bichos in sorteios:
        if len(bichos) >= 2:
            # Ordena alfabeticamente para evitar que ('TIGRE', 'VEADO') e ('VEADO', 'TIGRE') fiquem separados
            for par in combinations(sorted(bichos), 2):
                lista_pares.append(par)
                
    if not lista_pares:
        return pd.Series(dtype=int)
    return pd.Series(lista_pares).value_counts()

print(f"👥 DETECTOR DE COOCORRÊNCIA DE CASAIS - HORÁRIO: {HORARIO_ALVO} 👥")
print("=" * 80)

for nome_janela, df_janela in janelas.items():
    print(f"\n{nome_janela}")
    print("-" * 80)
    
    contagem_casais = calcular_frequencia_casais(df_janela)
    
    if contagem_casais.empty:
        print("   -> Nenhum par de bicho se repetiu no mesmo sorteio nesta janela.")
        continue
        
    # Extrai os limites estatísticos para classificar a temperatura dos casais
    max_aparições = contagem_casais.max()
    
    quentes, mornos, frios = [], [], []
    
    for par, qtd in contagem_casais.items():
        par_formatado = f"• {par[0]} & {par[1]} ({qtd}x)"
        
        # Classificação baseada no teto de aparições da própria janela
        if max_aparições > 1:
            if qtd == max_aparições:
                quentes.append(par_formatado)
            elif qtd > 1:
                mornos.append(par_formatado)
            else:
                frios.append(par_formatado)
        else:
            # Se todos saíram apenas 1 vez, entram como mornos em observação
            mornos.append(par_formatado)
            
    print(f"   🔥 CASAIS QUENTES (Saindo grudados): {', '.join(quentes[:3]) if quentes else 'Nenhum par dominante ainda'}")
    print(f"   🫖 CASAIS MORNOS  (Frequência regular): {', '.join(mornos[:4]) if mornos else 'Nenhum'}")
    print(f"   ❄️ CASAIS FRIOS   (Saíram apenas 1x) : {len(frios)} pares listados na base")
print("=" * 80)

👥 DETECTOR DE COOCORRÊNCIA DE CASAIS - HORÁRIO: 17:00 👥

⚡ JANELA 24H MÓVEIS
--------------------------------------------------------------------------------
   🔥 CASAIS QUENTES (Saindo grudados): Nenhum par dominante ainda
   🫖 CASAIS MORNOS  (Frequência regular): • BORBOLETA & CABRA (1x), • BORBOLETA & CAMELO (1x), • BORBOLETA & COELHO (1x), • BORBOLETA & MACACO (1x)
   ❄️ CASAIS FRIOS   (Saíram apenas 1x) : 0 pares listados na base

🔄 JANELA 48H MÓVEIS
--------------------------------------------------------------------------------
   🔥 CASAIS QUENTES (Saindo grudados): • BORBOLETA & MACACO (2x)
   🫖 CASAIS MORNOS  (Frequência regular): Nenhum
   ❄️ CASAIS FRIOS   (Saíram apenas 1x) : 34 pares listados na base

⏳ JANELA 72H MÓVEIS
--------------------------------------------------------------------------------
   🔥 CASAIS QUENTES (Saindo grudados): • BORBOLETA & ELEFANTE (2x), • BORBOLETA & MACACO (2x)
   🫖 CASAIS MORNOS  (Frequência regular): Nenhum
   ❄️ CASAIS FRIOS   (Saíram ape

In [47]:
# ==============================================================================
# 🔮 ETAPA 3: RASTREADOR DE MILHARES E CENTENAS (QUENTES, MORNOS E FRIOS)
# ==============================================================================
import pandas as pd
from sqlalchemy import select
from datetime import date, timedelta
from database import AsyncSessionLocal
from models import ResultadoLoteria, BichoGrupoDezena

HORARIO_ALVO = '23:00'
LOTERIA_ALVO = 'Nacional'

# 1. Puxa tabelas vivas do banco de dados NeonDB
async with AsyncSessionLocal() as session:
    bichos_query = await session.execute(select(BichoGrupoDezena))
    tabela_bichos_db = {b.grupo: b.bicho for b in bichos_query.scalars().all()}
    
    res = await session.execute(select(ResultadoLoteria))
    df_raw = pd.DataFrame([item.__dict__ for item in res.scalars().all()])

# 2. Modelagem e Higienização completa dos dados na RAM (Evita KeyError)
df_raw['data_hora'] = pd.to_datetime(df_raw['data_hora'])
df_raw['data_sorteio'] = df_raw['data_hora'].dt.date  # <--- Linha salvadora que corrige o erro!
df_raw['dezena'] = df_raw['resultado'].astype(str).str.zfill(2).str[-2:]
df_raw['grupo'] = df_raw['grupo'].fillna(
    df_raw['resultado'].astype(str).str.zfill(2).str[-2:].apply(
        lambda d: 25 if int(d) == 0 else ((int(d) - 1) // 4) + 1 if d.isdigit() else None
    )
)
df_raw['bicho'] = df_raw['grupo'].map(tabela_bichos_db)

# Filtra estritamente a loteria e o horário alvo para análise cirúrgica
df_alvo = df_raw[(df_raw['no_loteria'] == LOTERIA_ALVO) & (df_raw['horario'] == HORARIO_ALVO)].copy()

# Extrai os últimos 3 dígitos (Centenas) e os 4 dígitos (Milhares)
df_alvo['centena'] = df_alvo['resultado'].astype(str).str.zfill(4).str[-3:]
df_alvo['milhar'] = df_alvo['resultado'].astype(str).str.zfill(4)

# Definição das variáveis de tempo móveis (Evita NameError)
hoje_analise = date.today()
ultimo_sorteio = df_raw['data_hora'].max()
marcador_48h = ultimo_sorteio - pd.Timedelta(hours=48)

janelas_mils = {
    "⚡ RADAR ULTRA-CURTO (Últimas 48h às 17h)": df_alvo[df_alvo['data_hora'] >= marcador_48h],
    "⏳ MATRIZ MACRO (Últimos 30 dias às 17h)": df_alvo[df_alvo['data_sorteio'] >= (hoje_analise - timedelta(days=30))]
}

print(f"🔮 RASTREADOR DE MILHARES E CENTENAS - HORÁRIO: {HORARIO_ALVO} ({LOTERIA_ALVO}) 🔮")
print("=" * 80)

for nome_j, df_j in janelas_mils.items():
    print(f"\n{nome_j}")
    print("-" * 80)
    
    if df_j.empty:
        print("   -> Dados insuficientes para processar volumes de milhares nesta janela.")
        continue
        
    # Análise de Centenas (Terminações de 3 dígitos)
    contagem_cen = df_j['centena'].value_counts()
    
    # Padrão de Inicial de Milhar (Captura a tendência do primeiro dígito da banca)
    df_j['primeiro_digito'] = df_j['milhar'].str[0]
    contagem_digito = df_j['primeiro_digito'].value_counts()
    
    print("   🎯 CENTENAS (3 Dígitos) MAIS RECORRENTES:")
    for cen, qtd in list(contagem_cen.head(3).items()):
        # Calcula dinamicamente o bicho da centena baseado nos dois últimos dígitos dela
        dezena_cen = cen[-2:]
        grupo_cen = 25 if int(dezena_cen) == 0 else ((int(dezena_cen) - 1) // 4) + 1 if dezena_cen.isdigit() else None
        bicho_cen = tabela_bichos_db.get(grupo_cen, "Desconhecido")
        print(f"      • Centena [{cen}] ➔ {qtd}x (Pertence ao bicho: {bicho_cen})")
        
    print("\n   🎰 COMPORTAMENTO DO 1º DÍGITO (Milhar inicial):")
    quentes_d = [f"[{d}] ({q}x)" for d, q in list(contagem_digito.head(2).items())]
    frios_d = [f"[{d}] ({q}x)" for d, q in list(contagem_digito.tail(2).items())]
    print(f"      🔥 Inicial Quente: {', '.join(quentes_d)}")
    print(f"      ❄️ Inicial Frio  : {', '.join(frios_d)}")
print("=" * 80)

🔮 RASTREADOR DE MILHARES E CENTENAS - HORÁRIO: 23:00 (Nacional) 🔮

⚡ RADAR ULTRA-CURTO (Últimas 48h às 17h)
--------------------------------------------------------------------------------
   🎯 CENTENAS (3 Dígitos) MAIS RECORRENTES:
      • Centena [371] ➔ 1x (Pertence ao bicho: PORCO)
      • Centena [683] ➔ 1x (Pertence ao bicho: TOURO)
      • Centena [290] ➔ 1x (Pertence ao bicho: URSO)

   🎰 COMPORTAMENTO DO 1º DÍGITO (Milhar inicial):
      🔥 Inicial Quente: [5] (4x), [1] (3x)
      ❄️ Inicial Frio  : [3] (2x), [8] (1x)

⏳ MATRIZ MACRO (Últimos 30 dias às 17h)
--------------------------------------------------------------------------------
   🎯 CENTENAS (3 Dígitos) MAIS RECORRENTES:
      • Centena [902] ➔ 3x (Pertence ao bicho: AVESTRUZ)
      • Centena [274] ➔ 2x (Pertence ao bicho: PAVÃO)
      • Centena [528] ➔ 2x (Pertence ao bicho: CARNEIRO)

   🎰 COMPORTAMENTO DO 1º DÍGITO (Milhar inicial):
      🔥 Inicial Quente: [0] (50x), [1] (23x)
      ❄️ Inicial Frio  : [8] (14x), [2